# Tencent WeMM-Embedding-9B + Qdrant — Kaggle T4×2 Production Demo

**VI:** Notebook public này giữ nguyên science/runtime authority đã được chấp nhận nhưng trình bày lại workflow theo từng phần rõ ràng để reviewer dễ đọc. Execution vẫn atomic: chỉ có **một code cell thực thi**, còn các cell Markdown bên dưới là presentation/documentation cells.

**EN:** This public notebook preserves the accepted science/runtime authority while restoring a sectioned notebook layout for reviewers. Execution remains atomic: there is **one executable code cell**, while the Markdown cells below are presentation/documentation cells.

## Kiến trúc, search spaces và Kaggle Inputs / Architecture, search spaces, and Kaggle Inputs

**Semantic corpus retrieval:** production Qdrant corpus với `99,967` entities mỗi collection / production Qdrant corpus with `99,967` entities per collection.

**Visual robustness retrieval:** temporary curated gallery gồm `4` original images / a temporary curated gallery containing `4` original images.

**Kaggle runtime requirements:**

- Accelerator: **NVIDIA T4 ×2**
- Internet: **ON**
- Dataset: `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots`, version `1`
- Model: `dangkhoa2016/tencent-wemm-embedding-9b`, Transformers/default/version `1`

Hai benchmark dùng search space khác nhau và luôn được báo cáo riêng; `68/68` chỉ là tổng số retrieval checks đã thực thi / The two benchmarks use different search spaces and are always reported separately; `68/68` is only the total number of executed retrieval checks.

## Steps 1/8–5/8 — Bootstrap + chuẩn bị hệ thống / Bootstrap + system setup

**VI:** Atomic runner checkout presentation source từ public release ref `v1.0.0`, sau đó bootstrap frozen science/runtime authority tại commit `d04bcd3e601b449b67d09ff1132cab965619d858`. Runtime bootstrap cài `requirements-kaggle.txt` và `requirements-demo.txt`, xác minh hardware/source/Dataset, reuse hoặc restore Qdrant và load GPU worker.

**EN:** The atomic runner checks out presentation source from public release ref `v1.0.0`, then bootstraps the frozen science/runtime authority at commit `d04bcd3e601b449b67d09ff1132cab965619d858`. The runtime bootstrap installs `requirements-kaggle.txt` and `requirements-demo.txt`, verifies hardware/source/Dataset, reuses or restores Qdrant, and loads the GPU worker.

Qdrant và GPU worker được giữ sống xuyên suốt Steps 6, 7A và 7B để tránh lifecycle sai lệch / Qdrant and the GPU worker remain live across Steps 6, 7A, and 7B to preserve lifecycle semantics.

## Step 6/8 — Truy xuất văn bản song ngữ / Bilingual text retrieval

**VI:** Giữ nguyên 5 frozen EN↔VI examples và 20 retrieval paths đã được chấp nhận. Presentation layer hiển thị **full query text**, **full candidate text**, tách rõ **TOP-1 WINNER / KẾT QUẢ #1** và **NEAREST COMPETITOR / ĐỐI THỦ GẦN NHẤT**; không clipping và không diễn giải raw cosine thành confidence percentage.

**EN:** Preserve the five frozen EN↔VI examples and 20 accepted retrieval paths. The presentation layer shows **full query text**, **full candidate text**, and clearly separates the **TOP-1 WINNER** from the **NEAREST COMPETITOR**; there is no clipping and raw cosine is never presented as a confidence percentage.

## Step 7A/8 — Truy xuất semantic ảnh→văn bản / Semantic image→text retrieval

**VI:** Giữ nguyên frozen 4-example image→text showcase trên production corpus `99,967` entities. Mỗi ảnh truy xuất entity text tiếng Anh và tiếng Việt ở cả `4096d` và `1024d`; raw cosine cross-modal được hiển thị nguyên bản.

**EN:** Preserve the frozen four-example image→text showcase over the production `99,967`-entity corpus. Each image retrieves the corresponding English and Vietnamese text entity at both `4096d` and `1024d`; cross-modal raw cosine is reported as-is.

## Step 7B/8 — Độ bền truy xuất hình ảnh / Visual robustness retrieval

**VI:** Temporary gallery gồm đúng 4 original P18 images đã curate. Mỗi entity chạy 4 transforms — resize `80%`, JPEG quality `90`, center crop `96%`, brightness `103%` — ở `4096d` và `1024d`, tổng cộng **32 retrieval paths**. PASS yêu cầu toàn bộ `32/32` rank #1 và raw cosine `>= 0.90`; không rescale và không nới threshold.

**EN:** A temporary gallery contains exactly four curated original P18 images. Each entity runs four transforms — resize `80%`, JPEG quality `90`, center crop `96%`, brightness `103%` — at `4096d` and `1024d`, for **32 retrieval paths** total. PASS requires all `32/32` paths to rank #1 with raw cosine `>= 0.90`; there is no rescaling and no threshold relaxation.

## Step 8/8 — Đóng phiên + nghiệm thu / Closeout + acceptance

**VI:** Closeout giải phóng GPU worker, xác minh VRAM reclaim, dừng và seal Qdrant production storage. Scorecard cuối giữ tách biệt semantic `36/36 TOP-1` trên corpus `99,967` entities và visual robustness `32/32 TOP-1` trên temporary 4-image gallery.

**EN:** Closeout releases the GPU worker, verifies VRAM reclaim, stops and seals production Qdrant storage. The final scorecard keeps semantic `36/36 TOP-1` over the `99,967`-entity corpus separate from visual robustness `32/32 TOP-1` over the temporary four-image gallery.

## Contract chấp nhận + cách chạy / Acceptance contract + how to run

**VI:** Chỉ cần chạy code cell duy nhất bên dưới trong fresh Kaggle kernel. Cell đó gọi `run_public_notebook()` và thực thi tuần tự bootstrap → setup → Step 6 → Step 7A → Step 7B → Step 8 trong **một Python submission**, nên không phụ thuộc Kaggle phải submit code cell kế tiếp.

**EN:** Run only the single code cell below in a fresh Kaggle kernel. It calls `run_public_notebook()` and executes bootstrap → setup → Step 6 → Step 7A → Step 7B → Step 8 within **one Python submission**, so execution does not depend on Kaggle submitting a subsequent code cell.

Expected terminal markers include:

- `PUBLIC_NOTEBOOK_PRESENTATION_REF=v1.0.0`
- `TEXT_SHOWCASE_ALL_TOP1=5/5`
- `SEMANTIC_CORPUS_RETRIEVAL_PATHS_TOP1=36/36`
- `VISUAL_ROBUSTNESS_RETRIEVAL_PATHS_TOP1=32/32`
- `PUBLIC_DEMO_TOTAL_EXECUTED_RETRIEVAL_CHECKS=68/68`
- `WORKER_LIFECYCLE_GPU_RECLAIM=PASS`
- `QDRANT_STORAGE_SEAL=PASS`
- `ATOMIC_NOTEBOOK_RUNNER=PASS`

In [ ]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys

REPO = "https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU.git"
PUBLIC_RELEASE_REF = "v1.0.0"
SOURCE_ROOT = Path("/kaggle/working/wemm-public-notebook-source-v1.0.0")

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)

subprocess.run(["git", "init", "-q"], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "remote", "add", "origin", REPO], cwd=SOURCE_ROOT, check=True)
subprocess.run(
    ["git", "fetch", "-q", "--depth", "1", "origin", PUBLIC_RELEASE_REF],
    cwd=SOURCE_ROOT,
    check=True,
)
subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=SOURCE_ROOT, check=True)

sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

import wemm_notebook
from wemm_notebook import run_public_notebook

presentation_file = Path(wemm_notebook.__file__).resolve()
assert SOURCE_ROOT.resolve() in presentation_file.parents, (presentation_file, SOURCE_ROOT)
print("PUBLIC_NOTEBOOK_PRESENTATION_SOURCE=PASS", flush=True)
print("PUBLIC_NOTEBOOK_PRESENTATION_REF=" + PUBLIC_RELEASE_REF, flush=True)

run_public_notebook()
